# Basic SHAP Interaction Value Example in XGBoost

This notebook shows how the SHAP interaction values for a very simple function are computed. We start with a simple linear function, and then add an interaction term to see how it changes the SHAP values and the SHAP interaction values.

In [ ]:
import numpy as np
import xgboost
from sklearn.linear_model import LinearRegression

import shap

## Explain a linear function with no interactions

Simulate some binary data and a linear outcome with an interaction term.

Note we make the features in `X` perfectly independent of each other to make it easy to solve for the exact SHAP values.

In [ ]:
N = 2_000
X = np.zeros((N, 5))

X[:1_000, 0] = 1

X[:500, 1] = 1
X[1_000:1_500, 1] = 1

X[:250, 2] = 1
X[500:750, 2] = 1
X[1_000:1_250, 2] = 1
X[1_500:1_750, 2] = 1

# mean-center the data
X[:, 0:3] -= 0.5

y = 2 * X[:, 0] - 3 * X[:, 1]

We see that the variables are indeed independent

In [ ]:
np.cov(X.T)

and mean-centered.

In [ ]:
X.mean(axis=0)

In [ ]:
# train a model with single tree
Xd = xgboost.DMatrix(X, label=y)
model = xgboost.train({"eta": 1, "max_depth": 3, "base_score": 0, "lambda": 0}, Xd, 1)

print("Model error =", np.linalg.norm(y - model.predict(Xd)))
print(model.get_dump(with_stats=True)[0])

### SHAP values

In [ ]:
pred = model.predict(Xd, output_margin=True)

explainer = shap.TreeExplainer(model)
explanation = explainer(Xd)

shap_values = explanation.values
# make sure the SHAP values add up to marginal predictions
np.abs(shap_values.sum(axis=1) + explanation.base_values - pred).max()

If we build a [beeswarm][beeswarm_doclink] plot, we see that only features 0 and 1 have any effect on the output, and that their effects only have two possible magnitudes (1/-1 and 1.5/-1.5 correspondingly).

[beeswarm_doclink]: ../../../generated/shap.plots.beeswarm.rst#shap.plots.beeswarm

In [ ]:
shap.plots.beeswarm(explanation)

Train a linear model

In [ ]:
lr = LinearRegression()
lr.fit(X, y)
lr_pred = lr.predict(X)
lr.coef_.round(2)

Make sure the computed SHAP values match the true SHAP values
(we can compute the true SHAP values directly for any linear regression)

In [ ]:
main_effect_shap_values = lr.coef_ * (X - X.mean(0))
np.linalg.norm(shap_values - main_effect_shap_values)

### SHAP interaction values

Note that when there are no interactions present, the SHAP interaction values are just a diagonal matrix with the SHAP values on the diagonal.

In [ ]:
shap_interaction_values = explainer.shap_interaction_values(Xd)
shap_interaction_values[0]

Let's ensure that the SHAP interaction values sum to the marginal predictions:

In [ ]:
np.abs(shap_interaction_values.sum((1, 2)) + explainer.expected_value - pred).max()

And ensure the main effects from the SHAP interaction values match those from a linear model:

In [ ]:
total = 0
for i in range(N):
    for j in range(5):
        total += np.abs(shap_interaction_values[i, j, j] - main_effect_shap_values[i, j])
total

## Explain a linear model with one interaction

Simulate some binary data and a linear outcome with an interaction term.

Note that we make the features in `X` perfectly independent of each other to make it easy to solve for the exact SHAP values.

In [ ]:
N = 2_000
X = np.zeros((N, 5))
X[:1_000, 0] = 1

X[:500, 1] = 1
X[1_000:1_500, 1] = 1

X[:250, 2] = 1
X[500:750, 2] = 1
X[1_000:1_250, 2] = 1
X[1_500:1_750, 2] = 1

X[:125, 3] = 1
X[250:375, 3] = 1
X[500:625, 3] = 1
X[750:875, 3] = 1
X[1_000:1_125, 3] = 1
X[1_250:1_375, 3] = 1
X[1_500:1_625, 3] = 1
X[1_750:1_875, 3] = 1

# we can't exactly mean center the data or XGBoost has trouble finding the splits
X[:, :4] -= 0.4999

# interaction of features is implemented as the multiplication of the features. Note that any other function of the
#  features would also work, but is harder to interpret (e.g. sin(x1*x2)).
y = 2 * X[:, 0] - 3 * X[:, 1] + 2 * X[:, 1] * X[:, 2]

In [ ]:
X.mean(axis=0)

In [ ]:
# train a model with single tree
Xd = xgboost.DMatrix(X, label=y)
model = xgboost.train({"eta": 1, "max_depth": 4, "base_score": 0, "lambda": 0}, Xd, 1)
print("Model error =", np.linalg.norm(y - model.predict(Xd)))
print(model.get_dump(with_stats=True)[0])

### SHAP values

In [ ]:
pred = model.predict(Xd, output_margin=True)

explainer = shap.TreeExplainer(model)
explanation = explainer(Xd)

shap_values = explanation.values
# make sure the SHAP values add up to marginal predictions
np.abs(shap_values.sum(axis=1) + explanation.base_values - pred).max()

If we build a [beeswarm][beeswarm_doclink] plot, we see that now only features 3 and 4 don't matter, and that feature 1 can have four possible effect sizes due to interactions.

[beeswarm_doclink]: ../../../generated/shap.plots.beeswarm.rst#shap.plots.beeswarm

In [ ]:
shap.plots.beeswarm(explanation)

Train a linear model

In [ ]:
lr = LinearRegression()
lr.fit(X, y)
lr_pred = lr.predict(X)
lr.coef_.round(2)

Note that the SHAP values no longer match the main effects because they now include interaction effects.

In [ ]:
main_effect_shap_values = lr.coef_ * (X - X.mean(0))
np.linalg.norm(shap_values - main_effect_shap_values)

### SHAP interaction values

SHAP interaction contributions are displayed on the off-diagonal

In [ ]:
shap_interaction_values = explainer.shap_interaction_values(Xd)
shap_interaction_values[0].round(2)

Ensure the SHAP interaction values sum to the marginal predictions

In [ ]:
np.abs(shap_interaction_values.sum((1, 2)) + explainer.expected_value - pred).max()

While the main effects no longer match the SHAP values when interactions are present, they do match the main effects (from the linear model) on the diagonal of the SHAP interaction value matrix.

In [ ]:
total = 0
for i in range(N):
    for j in range(5):
        total += np.abs(shap_interaction_values[i, j, j] - main_effect_shap_values[i, j])
total

In [ ]:
shap.dependence_plot(0, shap_values, X)

If we build a dependence plot for feature 0, we see that it only takes two values and that these values are entirely dependent on the value of the feature. Hence, they lie on a straight line (the value of feature 0 entirely determines its effect because it has no interactions with other features).

In contrast, if we build a dependence plot for feature 2, we see that it takes 4 possible values and they are not entirely determined by the value of feature 2.
Instead they also depend on the value of feature 3.
This vertical spread in a dependence plot represents the effects of non-linear interactions.

In [ ]:
shap.dependence_plot(2, shap_values, X)